# Train

## import

In [ ]:
import pandas as pd

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
import numpy as np

In [ ]:
from sklearn.model_selection import LearningCurveDisplay, ShuffleSplit

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

In [ ]:
sns.set()

In [ ]:
import optuna

## Load

In [ ]:
df = pd.read_csv("../data/data.csv", dtype="str")

In [ ]:
df = df.astype({
    "values": float,
    "site_area": float,
    "total_floor_area": float,
    "minutes": int,
    "time_to_ib": float,
    "is_express": bool
})

In [ ]:
df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[["site_area", "total_floor_area", "minutes", "time_to_ib", "is_express"]],
    df["values"],
    test_size=0.2,
    random_state=0
)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

# HyperparameterOptimization

In [ ]:
def objective(trial: optuna.Trial, X=X_train, y=y_train):
    X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, random_state=0)
    n_estimators = trial.suggest_int("n_estimators", 10, 150)
    regr = RandomForestRegressor(n_estimators=n_estimators)
    regr.fit(X_train, y_train)
    y_pred = regr.predict(X_val)
    error = np.sqrt(mean_squared_error(y_true=y_val, y_pred=y_pred))
    return error

In [ ]:
study = optuna.create_study()
study.optimize(objective, n_trials=100)

In [ ]:
study.best_params

In [ ]:
study.best_value

## LearningCurve

In [ ]:
regr = RandomForestRegressor(n_estimators=121)

In [ ]:
common_params = {
    "X": X_train,
    "y": y_train,
    "train_sizes": np.linspace(0.1, 1.0, 10),
    "score_type": "both",
    "scoring": "neg_root_mean_squared_error",
    "line_kw": {"marker": "o"},
    "std_display_style": "fill_between",
    "score_name": "Negative RMSE",
}

In [ ]:
_, ax = plt.subplots(figsize=(10, 5))

LearningCurveDisplay.from_estimator(regr, **common_params, ax=ax)
handles, label = ax.get_legend_handles_labels()
ax.legend(handles[:2], ["Training Score", "Test Score"])
ax.set_title(f"Learning Curve for {regr.__class__.__name__}");

## Train

In [ ]:
regr.fit(X_train, y_train)

In [ ]:
np.sqrt(mean_squared_error(y_true=y_train, y_pred=regr.predict(X_train)))

In [ ]:
np.sqrt(mean_squared_error(y_true=y_test, y_pred=regr.predict(X_test)))